# 2hClub — run the ball detector (Colab)

Runs the trained detector (`best.pt`) over the tactical video and saves candidates per frame to Drive.
Same method as `vision/detect.py`: 8 overlapping 640 tiles per frame, top 5 candidates kept.

1. Upload `BAR-ATH_tactical.mp4` **anywhere in Google Drive** (5 GB; wait until the upload is complete).
   `best.pt` is already there from training.
2. Runtime → Change runtime type → **GPU**.
3. Runtime → **Run all**.

**Resumable:** each span is saved to Drive as soon as it is done, and finished spans are skipped.
If Colab disconnects, reconnect and Run all again — it carries on where it stopped.

Keep the video and outputs **private**: they are match data (RULES.md, terms of data use).

In [ ]:
# What to run. 'windows' = the public window + our 6 labelled windows (for scoring).
# 'match' = the whole match in 10,000-frame chunks (for the final submission, later).
MODE = 'windows'
OUT_SUBDIR = '2hclub/detections'
IMGSZ = 1280            # must match training

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics==8.4.190
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import glob, os, shutil, time
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} not found anywhere in your Google Drive'
    print('found', hits[0], f'({os.path.getsize(hits[0]) / 1e9:.2f} GB)')
    return hits[0]
VIDEO_DRIVE = find('BAR-ATH_tactical.mp4')
WEIGHTS = find('runs/ball_r1/weights/best.pt')
# Reading 5 GB from Drive frame by frame is slow; copy it to the Colab disk once (~2-4 min).
VIDEO = '/content/BAR-ATH_tactical.mp4'
if not os.path.exists(VIDEO) or os.path.getsize(VIDEO) != os.path.getsize(VIDEO_DRIVE):
    shutil.copy(VIDEO_DRIVE, VIDEO)
OUT = os.path.join('/content/drive/MyDrive', OUT_SUBDIR)
os.makedirs(OUT, exist_ok=True)

In [ ]:
N_FRAMES = 199319
MARGIN = 250   # context either side of a window, for the tracker that runs on these later
WINDOWS = {'public': 11185, 'w1': 35000, 'w2': 60000, 'w3': 82000,
           'w4': 120000, 'w5': 145000, 'w6': 175000}   # each 7,500 frames long
if MODE == 'windows':
    SPANS = [(max(0, s - MARGIN), min(N_FRAMES, s + 7500 + MARGIN)) for s in WINDOWS.values()]
else:
    SPANS = [(s, min(N_FRAMES, s + 10000)) for s in range(0, N_FRAMES, 10000)]
todo = [sp for sp in SPANS if not os.path.exists(f'{OUT}/det_{sp[0]}_{sp[1]}.npz')]
print(len(SPANS), 'spans,', len(todo), 'still to do,', sum(b - a for a, b in todo), 'frames')

In [ ]:
# Mirrors vision/detect.py -- keep the constants identical.
import cv2, numpy as np
from ultralytics import YOLO
TILE, TILE_X, TILE_Y = 640, (0, 427, 853, 1280), (0, 440)
MIN_CONF, MERGE_PX, MAX_PER_FRAME = 0.05, 8, 5
model = YOLO(WEIGHTS)
import logging; logging.getLogger('ultralytics').setLevel(logging.ERROR)   # silence a per-call 'half' warning

def merge(points):
    kept = []
    for x, y, conf in sorted(points, key=lambda p: -p[2]):
        if all(np.hypot(x - kx, y - ky) >= MERGE_PX for kx, ky, _ in kept):
            kept.append((x, y, conf))
    return kept[:MAX_PER_FRAME]

def detect_frame(img):
    origins = [(x0, y0) for y0 in TILE_Y for x0 in TILE_X]
    tiles = [img[y0:y0 + TILE, x0:x0 + TILE] for x0, y0 in origins]
    results = model.predict(tiles, imgsz=IMGSZ, conf=MIN_CONF, half=True, verbose=False)
    points = []
    for (x0, y0), r in zip(origins, results):
        for (cx, cy, _, _), conf in zip(r.boxes.xywh.cpu().numpy(), r.boxes.conf.cpu().numpy()):
            points.append((float(cx + x0), float(cy + y0), float(conf)))
    return merge(points)

In [ ]:
for start, end in SPANS:
    path = f'{OUT}/det_{start}_{end}.npz'
    if os.path.exists(path):
        continue
    t = time.time()
    dets = np.full((end - start, MAX_PER_FRAME, 3), np.nan, dtype=np.float32)
    cap = cv2.VideoCapture(VIDEO)
    cap.set(cv2.CAP_PROP_POS_FRAMES, start)
    for i in range(end - start):
        ok, img = cap.read()
        if not ok:
            break
        for j, p in enumerate(detect_frame(img)):
            dets[i, j] = p
    np.savez_compressed('/content/tmp.npz', start=start, dets=dets)
    shutil.copy('/content/tmp.npz', path)           # only complete spans ever land in Drive
    print(f'{start}-{end}: {(end - start) / (time.time() - t):.1f} fps, saved', flush=True)
print('all spans done ->', OUT)